# Synthetic Sales Data — Analysis

Picks up where `quality_report.json` ends: surfaces the validator's verdict, visualizes the distributions, tours the domain, and runs a broader set of integrity checks the validator doesn't cover.

**Inputs:** `data/raw/sales_1k.csv`, `data/raw/quality_report.json`, `data/sample/synthetic_config.json`. Generate them first with `uv run python -m src.datagen.generate` if missing.

In [ ]:
import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.datagen.config import load_config
from src.sales_data.metadata.column_definitions import EXPECTED_COLUMNS, ORDER_FIELDS
from src.sales_data.metadata.enums import Channel, ProductCategory, Region

CSV_PATH = PROJECT_ROOT / "data" / "raw" / "sales_1k.csv"
REPORT_PATH = PROJECT_ROOT / "data" / "raw" / "quality_report.json"
CONFIG_PATH = PROJECT_ROOT / "data" / "sample" / "synthetic_config.json"

for p, hint in [(CSV_PATH, "uv run python -m src.datagen.generate"),
                (REPORT_PATH, "uv run python -m src.datagen.generate"),
                (CONFIG_PATH, "cp data/sample/synthetic_config.sample.json data/sample/synthetic_config.json")]:
    assert p.exists(), f"missing: {p}\nrun: {hint}"

df = pd.read_csv(CSV_PATH, parse_dates=["order_date"])
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
cfg = load_config(CONFIG_PATH)

print(f"rows:        {df.shape[0]}")
print(f"columns:     {df.shape[1]}")
print(f"cfg.seed:    {cfg.seed}")
print(f"date_range:  {cfg.date_start} .. {cfg.date_end}")
print(f"summary:     {report['summary']}")

## 1. Quality — what the validator reports

Schema conformance (column set, dtypes, non-null required fields, `order_id` uniqueness) plus distributional checks (χ² goodness-of-fit on categorical mixes, per-row band/range violation rates). Pass/fail gates come from `cfg.tolerances`.

In [ ]:
s = report["summary"]
headline = pd.DataFrame(
    [
        ("overall",      s["pass"]),
        ("schema",       s["schema_pass"]),
        ("distribution", s["distribution_pass"]),
        ("row_count",    s["row_count"]),
        ("generated_at", s["generated_at"]),
    ],
    columns=["check", "value"],
)
headline

In [ ]:
rows = []
for name, d in report["distributions"].items():
    if d.get("chi2") is not None:
        metric = f"chi2={d['chi2']:.3f}, p={d['p_value']:.3f}"
    elif "violation_rate" in d:
        metric = f"violation_rate={d['violation_rate']:.4f} (max {d['max_rate']})"
    elif "out_of_range_rate" in d:
        metric = f"out_of_range_rate={d['out_of_range_rate']:.4f}"
    else:
        metric = "-"
    rows.append((name, metric, "PASS" if d["passed"] else "FAIL"))

print(f"tolerance: chi2_p_min={cfg.tolerances.distribution_chi2_p_min}, band_violation_rate_max={cfg.tolerances.band_violation_rate_max}\n")
pd.DataFrame(rows, columns=["check", "metric", "verdict"])

## 2. Distribution — observed vs expected

The generator draws region / channel / category from normalized weights, prices uniformly within per-category bands, and quantities from a clipped Poisson. These plots confirm the samples track the configured shapes.

In [ ]:
def _expected_share(weights: dict[str, float]) -> dict[str, float]:
    total = sum(weights.values())
    return {k: v / total for k, v in weights.items()}

categorical = [
    ("region",   df["region"].value_counts(normalize=True),   _expected_share(cfg.region_weights)),
    ("channel",  df["channel"].value_counts(normalize=True),  _expected_share(cfg.channel_weights)),
    ("category", df["category"].value_counts(normalize=True), _expected_share(cfg.category_weights)),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (name, observed, expected) in zip(axes, categorical):
    keys = list(expected.keys())
    x = np.arange(len(keys))
    ax.bar(x - 0.2, [observed.get(k, 0) for k in keys], width=0.4, label="observed", color="#4C72B0")
    ax.bar(x + 0.2, [expected[k] for k in keys],       width=0.4, label="expected", color="#DD8452")
    ax.set_xticks(x)
    ax.set_xticklabels(keys, rotation=20, ha="right")
    ax.set_title(name)
    ax.set_ylabel("share")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
categories = [c.value for c in ProductCategory]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, cat in zip(axes, categories):
    prices = df.loc[df["category"] == cat, "unit_price"]
    lo, hi = cfg.price_bands[cat]
    ax.hist(prices, bins=20, color="#4C72B0", edgecolor="white")
    ax.axvline(lo, color="#C44E52", linestyle="--", linewidth=1, label=f"band: {lo}-{hi}")
    ax.axvline(hi, color="#C44E52", linestyle="--", linewidth=1)
    ax.set_title(f"{cat}  (n={len(prices)}, mean={prices.mean():.2f})")
    ax.set_xlabel("unit_price")
    ax.legend(fontsize=8, loc="upper right")
axes[0].set_ylabel("orders")
plt.suptitle("Price distribution per category (uniform within band)", y=1.02, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, cat in zip(axes, categories):
    qtys = df.loc[df["category"] == cat, "quantity"]
    qmin, qmax, qmean = cfg.quantity_params[cat]
    bins = np.arange(qmin, qmax + 2) - 0.5
    ax.hist(qtys, bins=bins, color="#55A868", edgecolor="white")
    ax.axvline(qmean, color="#C44E52", linestyle="--", linewidth=1, label=f"poisson mean={qmean}")
    ax.set_title(f"{cat}  (clip {qmin}-{qmax}, mean obs={qtys.mean():.2f})")
    ax.set_xlabel("quantity")
    ax.legend(fontsize=8, loc="upper right")
axes[0].set_ylabel("orders")
plt.suptitle("Quantity distribution per category (clipped Poisson)", y=1.02, fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
daily = df.groupby(df["order_date"].dt.date).size()
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(daily.index, daily.values, color="#4C72B0", linewidth=0.8)
ax.axhline(daily.mean(), color="#C44E52", linestyle="--", linewidth=1, label=f"mean: {daily.mean():.2f}/day")
ax.set_title(f"Daily order volume (range {cfg.date_start} .. {cfg.date_end}, uniform day-offset)")
ax.set_xlabel("order_date")
ax.set_ylabel("orders")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 3. Domain shape

How the data looks as a sales dataset: catalog cardinality, per-order economics, customer activity, revenue concentration by category.

In [ ]:
expected_products = cfg.n_products_per_category * len(list(ProductCategory))
n_days = (cfg.date_end - cfg.date_start).days + 1
cardinality = pd.DataFrame(
    [
        ("customers",     df["customer_id"].nunique(), cfg.n_customers),
        ("products",      df["product_id"].nunique(),  expected_products),
        ("categories",    df["category"].nunique(),    len(list(ProductCategory))),
        ("regions",       df["region"].nunique(),      len(list(Region))),
        ("channels",      df["channel"].nunique(),     len(list(Channel))),
        ("order_dates",   df["order_date"].dt.date.nunique(), n_days),
    ],
    columns=["dimension", "observed_distinct", "expected_distinct"],
)
cardinality

In [ ]:
df = df.assign(revenue=lambda d: d["quantity"] * d["unit_price"])

per_order = df[["quantity", "unit_price", "revenue"]].describe().round(2)
per_category = df.groupby("category").agg(
    orders=("order_id", "count"),
    mean_quantity=("quantity", "mean"),
    mean_unit_price=("unit_price", "mean"),
    total_revenue=("revenue", "sum"),
).round(2)

print("Per-order economics (whole dataset):")
display(per_order)
print("\nPer-category economics:")
display(per_category)

In [ ]:
orders_per_customer = df.groupby("customer_id").size()
revenue_by_cat = df.groupby("category")["revenue"].sum().sort_values(ascending=False)
revenue_share = (revenue_by_cat / revenue_by_cat.sum() * 100).round(1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))

ax1.hist(orders_per_customer.values, bins=20, color="#4C72B0", edgecolor="white")
ax1.axvline(orders_per_customer.mean(), color="#C44E52", linestyle="--",
            label=f"mean: {orders_per_customer.mean():.1f}")
ax1.set_title(f"Orders per customer (n={len(orders_per_customer)} customers)")
ax1.set_xlabel("orders")
ax1.set_ylabel("customers")
ax1.legend(fontsize=9)

ax2.bar(revenue_by_cat.index, revenue_by_cat.values, color="#55A868", edgecolor="white")
for i, (cat, val) in enumerate(revenue_by_cat.items()):
    ax2.text(i, val, f"{revenue_share[cat]}%", ha="center", va="bottom", fontsize=9)
ax2.set_title("Total revenue by category")
ax2.set_ylabel("revenue")

plt.tight_layout()
plt.show()

print("\nTop 10 customers by order count:")
display(orders_per_customer.sort_values(ascending=False).head(10).rename("orders").to_frame())

## 4. Integrity checks (beyond the validator)

Soft properties the validator doesn't gate on: ID format regex, referential one-to-one bindings between IDs and names/categories, enum membership, positivity of numeric fields, date-range bounds.

In [ ]:
checks: list[tuple[str, bool, str]] = []

def _check(name: str, ok: bool, detail: str = "") -> None:
    checks.append((name, bool(ok), detail))

_check("order_id format ^ORD-\\d{6}$",
       df["order_id"].str.fullmatch(r"ORD-\d{6}").all(),
       f"violations: {(~df['order_id'].str.fullmatch(r'ORD-\d{6}')).sum()}")
_check("order_id unique",
       df["order_id"].is_unique,
       f"distinct: {df['order_id'].nunique()}/{len(df)}")
_check("customer_id format ^CUST-\\d{4}$",
       df["customer_id"].str.fullmatch(r"CUST-\d{4}").all())
_check("product_id format ^PROD-(ELE|STA|FUR)-\\d{3}$",
       df["product_id"].str.fullmatch(r"PROD-(ELE|STA|FUR)-\d{3}").all())

_check("customer_id -> customer_name 1:1",
       df.groupby("customer_id")["customer_name"].nunique().max() == 1)
_check("product_id -> product_name 1:1",
       df.groupby("product_id")["product_name"].nunique().max() == 1)
_check("product_id -> category 1:1",
       df.groupby("product_id")["category"].nunique().max() == 1)

region_values   = {r.value for r in Region}
channel_values  = {c.value for c in Channel}
category_values = {c.value for c in ProductCategory}
_check("region in enum",   set(df["region"].unique())   <= region_values)
_check("channel in enum",  set(df["channel"].unique())  <= channel_values)
_check("category in enum", set(df["category"].unique()) <= category_values)

required = [f.name for f in ORDER_FIELDS if not f.nullable]
nulls_by_col = df[required].isna().sum()
_check("no nulls in required cols",
       int(nulls_by_col.sum()) == 0,
       ", ".join(f"{c}={v}" for c, v in nulls_by_col.items() if v))

_check("quantity > 0",   bool((df["quantity"]   > 0).all()), f"min={int(df['quantity'].min())}")
_check("unit_price > 0", bool((df["unit_price"] > 0).all()), f"min={df['unit_price'].min()}")

start_ts = pd.Timestamp(cfg.date_start)
end_ts = pd.Timestamp(cfg.date_end)
in_range = df["order_date"].between(start_ts, end_ts).all()
_check("order_date in [start, end]", in_range,
       f"observed: {df['order_date'].min().date()} .. {df['order_date'].max().date()}")

table = pd.DataFrame(checks, columns=["check", "passed", "detail"])
table["verdict"] = table["passed"].map({True: "PASS", False: "FAIL"})
summary_line = f"{table['passed'].sum()}/{len(table)} passed"
print(summary_line)
table[["check", "verdict", "detail"]]

## 5. Summary

The current 1k run passes every check on three layers: the validator's quality gate, the shape-conformance plots against the configured weights/bands, and the broader integrity table. Design rationale lives in `docs/superpowers/specs/2026-05-12-synthetic-sales-data-design.md`.